# Sales Forecast — Visualization

Read-only charts from Unity Catalog **`serve`** (gold features) and **`ml`** (forecast outputs). Does **not** retrain models.

**Prerequisites**
1. Gold pipeline has run (`serve.*` feature tables exist)
2. `Sales Forecast Training.ipynb` has run at least once (`ml.*` tables populated)
3. Run **cell 1** first if Prophet components chart is needed (installs `prophet`)

**Charts**
- Historical sales trend (order count + quantity)
- Company forecast + 95% confidence band
- Holdout backtest (actual vs predicted)
- Prophet trend & seasonality components
- Exogenous driver panels
- Top-SKU material forecasts
- SKU wMAPE bar chart

In [ ]:
# Only required for the Prophet components section (cell 4). Safe to skip if you skip that section.
%pip install prophet
dbutils.library.restartPython()

In [ ]:
import json

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

CATALOG = "jm_databricks_learning_ws"
SERVE = f"{CATALOG}.serve"
ML = f"{CATALOG}.ml"

TARGET_COLUMN = "sales_quantity"
DEFAULT_REGRESSORS = (
    "purchase_quantity",
    "production_output_quantity",
    "inventory_on_hand",
    "downtime_hours",
)

plt.rcParams["figure.figsize"] = (14, 6)


def _table(name: str) -> pd.DataFrame:
    return spark.table(name).toPandas()


def latest_run_id(grain: str) -> int | None:
    rows = spark.sql(
        f"SELECT run_id FROM {ML}.forecast_runs WHERE grain = '{grain}' ORDER BY run_id DESC LIMIT 1"
    ).collect()
    return int(rows[0]["run_id"]) if rows else None


def plot_history_and_forecast(history, forecast, *, value_col, title, ylabel, figsize=(14, 6)):
    cutoff = pd.to_datetime(history["month_start_date"]).max()
    hist = history.copy()
    hist["ds"] = pd.to_datetime(hist["month_start_date"])
    fc = forecast.copy()
    fc["ds"] = pd.to_datetime(fc["forecast_month"])

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(hist["ds"], hist[value_col], label="Historical", color="steelblue", marker="o", ms=4, linewidth=1.5)
    ax.plot(fc["ds"], fc["yhat"], label="Forecast", color="tomato", linewidth=2)
    ax.fill_between(fc["ds"], fc["yhat_lower"], fc["yhat_upper"], alpha=0.2, color="tomato", label="95% CI")
    ax.axvline(cutoff, linestyle="--", color="gray", linewidth=1.2, label="Forecast Start")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Month")
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    fig.autofmt_xdate(rotation=30)
    plt.tight_layout()
    display(fig)
    plt.close(fig)


def plot_backtest(detail, *, title, ylabel):
    bt = detail.copy()
    bt["forecast_month"] = pd.to_datetime(bt["forecast_month"])
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(bt["forecast_month"], bt["actual"], marker="o", label="Actual", color="steelblue", linewidth=1.8)
    ax.plot(bt["forecast_month"], bt["predicted"], marker="s", label="Predicted (holdout)", color="tomato", linewidth=1.8)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Month")
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    fig.autofmt_xdate(rotation=30)
    plt.tight_layout()
    display(fig)
    plt.close(fig)


company_run_id = latest_run_id("company")
material_run_id = latest_run_id("material")

if company_run_id is None:
    raise RuntimeError(f"No company run in {ML}.forecast_runs — run Sales Forecast Training.ipynb first")

print(f"Company run_id : {company_run_id}")
print(f"Material run_id: {material_run_id}")

## 1. Historical sales trend

In [ ]:
trend = _table(f"{SERVE}.sales_trend_monthly")
trend["month_start_date"] = pd.to_datetime(trend["month_start_date"])

print(f"Shape      : {trend.shape}")
print(f"Date range : {trend['month_start_date'].min().date()} → {trend['month_start_date'].max().date()}")
display(trend[["order_count", "total_quantity"]].describe().round(0))

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(trend["month_start_date"], trend["order_count"], marker="o", ms=4, linewidth=1.8, color="steelblue")
axes[0].set_title("Monthly Order Count", fontweight="bold")
axes[0].set_ylabel("Order Count")
axes[0].grid(True, alpha=0.3)

axes[1].plot(trend["month_start_date"], trend["total_quantity"], marker="o", ms=4, linewidth=1.8, color="darkorange")
axes[1].set_title("Monthly Total Quantity", fontweight="bold")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Total Quantity")
axes[1].grid(True, alpha=0.3)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate(rotation=30)
plt.tight_layout()
display(fig)
plt.close(fig)

## 2. Company forecast — historical + next 12 months

In [ ]:
company_hist = _table(f"{SERVE}.company_forecast_features_monthly")
company_fc = spark.sql(f"""
    SELECT forecast_month, year_month, yhat, yhat_lower, yhat_upper
    FROM {ML}.sales_forecast_monthly
    WHERE run_id = {company_run_id} AND grain = 'company'
    ORDER BY forecast_month
""").toPandas()

plot_history_and_forecast(
    company_hist,
    company_fc,
    value_col=TARGET_COLUMN,
    title="Monthly Sales Quantity — Historical & 12-Month Forecast",
    ylabel="Sales Quantity",
)

next12 = company_fc.copy()
next12[["yhat", "yhat_lower", "yhat_upper"]] = next12[["yhat", "yhat_lower", "yhat_upper"]].round(0).astype(int)
display(next12.rename(columns={
    "year_month": "Month", "yhat": "Forecast", "yhat_lower": "Lower", "yhat_upper": "Upper"
})[["Month", "Forecast", "Lower", "Upper"]])

## 3. Holdout backtest (last 6 months)

In [ ]:
company_bt = spark.sql(f"""
    SELECT forecast_month, actual, predicted, error
    FROM {ML}.forecast_backtest_detail
    WHERE run_id = {company_run_id} AND grain_id = 'COMPANY'
    ORDER BY forecast_month
""").toPandas()

company_metrics = spark.sql(f"""
    SELECT metric_name, metric_value
    FROM {ML}.forecast_backtest
    WHERE run_id = {company_run_id} AND grain_id = 'COMPANY'
""").toPandas()

display(company_metrics.set_index("metric_name").round(2))

plot_backtest(
    company_bt,
    title="Company Holdout — Actual vs Predicted Sales Quantity",
    ylabel="Sales Quantity",
)

comparison = company_bt.copy()
comparison["forecast_month"] = pd.to_datetime(comparison["forecast_month"]).dt.strftime("%Y-%m")
display(comparison.round(1).rename(columns={
    "forecast_month": "Month", "actual": "Actual", "predicted": "Predicted", "error": "Error"
}))

## 4. Prophet components (trend & yearly seasonality)

Refits the company Prophet model for decomposition (requires cell 1 / `prophet` installed).

In [ ]:
from prophet import Prophet

run_meta = spark.sql(f"""
    SELECT params_json, horizon_months
    FROM {ML}.forecast_runs
    WHERE run_id = {company_run_id}
""").toPandas().iloc[0]

params = json.loads(run_meta["params_json"])
regressors = params.get("regressors", list(DEFAULT_REGRESSORS))
horizon = int(run_meta["horizon_months"])

features = company_hist.copy()
features["ds"] = pd.to_datetime(features["month_start_date"])
features["y"] = features[TARGET_COLUMN].astype(float)

active = [
    c for c in regressors
    if c in features.columns and features[c].astype(float).std() > 1e-9
]

model = Prophet(
    seasonality_mode=params.get("seasonality_mode", "multiplicative"),
    yearly_seasonality=params.get("yearly_seasonality", True),
    weekly_seasonality=params.get("weekly_seasonality", False),
    daily_seasonality=params.get("daily_seasonality", False),
    changepoint_prior_scale=params.get("changepoint_prior_scale", 0.1),
    seasonality_prior_scale=params.get("seasonality_prior_scale", 10.0),
)
for col in active:
    model.add_regressor(col)

model.fit(features[["ds", "y", *active]])
future = model.make_future_dataframe(periods=horizon, freq="MS")

for col in active:
    hist_map = features.set_index("ds")[col]
    vals = []
    for ds in future["ds"]:
        if ds in hist_map.index:
            vals.append(float(hist_map.loc[ds]))
        else:
            lag = ds - pd.DateOffset(years=1)
            if lag in hist_map.index:
                vals.append(float(hist_map.loc[lag]))
            else:
                trailing = hist_map[hist_map.index < ds].tail(3)
                vals.append(float(trailing.mean()) if len(trailing) else 0.0)
    future[col] = vals

forecast_full = model.predict(future)
fig2 = model.plot_components(forecast_full)
plt.suptitle("Forecast Components (Trend & Yearly Seasonality)", y=1.02, fontsize=12)
plt.tight_layout()
display(fig2)
plt.close(fig2)

## 5. Exogenous drivers (company level)

In [ ]:
drivers = company_hist.copy()
drivers["month_start_date"] = pd.to_datetime(drivers["month_start_date"])

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
series = [
    ("purchase_quantity", "Purchase Quantity", "seagreen"),
    ("production_output_quantity", "Production Output", "purple"),
    ("inventory_on_hand", "Inventory On Hand", "saddlebrown"),
    ("downtime_hours", "Downtime Hours", "crimson"),
]
for ax, (col, label, color) in zip(axes.ravel(), series):
    ax.plot(drivers["month_start_date"], drivers[col], marker="o", ms=3, color=color, linewidth=1.5)
    ax.set_title(label, fontweight="bold")
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

fig.suptitle("Company Exogenous Features Used in Forecasting", fontsize=13, fontweight="bold", y=1.02)
fig.autofmt_xdate(rotation=30)
plt.tight_layout()
display(fig)
plt.close(fig)

## 6. Top materials — forecast charts

Prefer **wMAPE** over MAPE for SKU accuracy (MAPE spikes when a month is near zero).

In [ ]:
if material_run_id is None:
    print("No material run found — run Sales Forecast Training.ipynb first.")
else:
    top = _table(f"{SERVE}.top_selling_materials")
    mat_metrics = spark.sql(f"""
        SELECT grain_id, metric_value AS wmape
        FROM {ML}.forecast_backtest
        WHERE run_id = {material_run_id} AND metric_name = 'wmape'
    """).toPandas()
    wmape = dict(zip(mat_metrics["grain_id"], mat_metrics["wmape"]))

    display(top[["sales_rank", "material_id", "material_name", "sold_quantity"]])

    for _, row in top.iterrows():
        mid = row["material_id"]
        name = row["material_name"]
        hist = spark.sql(f"""
            SELECT * FROM {SERVE}.forecast_features_monthly
            WHERE material_id = '{mid}' ORDER BY month_start_date
        """).toPandas()
        fc = spark.sql(f"""
            SELECT forecast_month, year_month, yhat, yhat_lower, yhat_upper
            FROM {ML}.sales_forecast_monthly
            WHERE run_id = {material_run_id} AND grain = 'material' AND grain_id = '{mid}'
            ORDER BY forecast_month
        """).toPandas()
        sku_wmape = float(wmape.get(mid, float("nan")))
        plot_history_and_forecast(
            hist,
            fc,
            value_col=TARGET_COLUMN,
            title=f"{mid} — {name}  |  holdout wMAPE: {sku_wmape:.1f}%",
            ylabel="Sales Quantity",
            figsize=(13, 4.5),
        )

## 7. Material backtest quality (bar chart)

In [ ]:
if material_run_id is not None:
    wmape_df = spark.sql(f"""
        SELECT b.grain_id AS material_id, b.metric_value AS wmape, t.material_name
        FROM {ML}.forecast_backtest b
        LEFT JOIN {SERVE}.top_selling_materials t ON b.grain_id = t.material_id
        WHERE b.run_id = {material_run_id} AND b.metric_name = 'wmape'
        ORDER BY b.metric_value
    """).toPandas()

    labels = wmape_df["material_id"] + "\n" + wmape_df["material_name"].fillna("").str.slice(0, 18)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(labels, wmape_df["wmape"], color="steelblue", edgecolor="white")
    ax.axhline(wmape_df["wmape"].median(), color="tomato", linestyle="--", label="Median wMAPE")
    ax.set_title("Top SKU Holdout wMAPE (%)", fontweight="bold")
    ax.set_ylabel("wMAPE %")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    display(fig)
    plt.close(fig)